**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Causal Inference

The question every regression dodges: what happens if we **intervene**? DAGs and d-separation, confounding and the backdoor adjustment, and the modern estimators — all on simulated worlds where we can *run the true intervention* and check the answer, the luxury real data never grants.

## 1. Pre-requisites

[Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb), [Independence](../Intro_Math/Analysis/Independence.ipynb); regression fluency ([Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
def ols(X, y):
    X1 = np.c_[np.ones(len(X)), X]
    return np.linalg.lstsq(X1, y, rcond=None)[0]

---
### 🕐 Session 1 of 3 — *Correlation, Confounding & the do-Operator* (~40 min)
**Goal:** watch a confounder manufacture a correlation; define intervention as graph surgery.
**Feeds into:** Session 2 (backdoor adjustment).

---

## 2. Seeing vs Doing

💡 **Intuition.** $P(Y \mid X = x)$ answers 'what do I expect of Y among units *observed* to have X = x?' — but observed X carries its causes with it. $P(Y \mid do(X = x))$ answers 'what if I *set* X to x?' — **graph surgery**: delete the arrows into X, because your intervention, not the world, chose it. The two differ exactly when a **confounder** feeds both X and Y. Our simulated world makes this concrete because we can *actually perform* the do — rerun the world with X forced.

In [2]:
# World 1: exercise (X) → health (Y), both driven by age (Z). TRUE causal effect: +2.0
def world(n, do_x=None):
    Z = rng.uniform(20, 70, n)                                # age (confounder)
    X = np.clip(8 - 0.1*Z + rng.standard_normal(n), 0, None)  # older people exercise less
    if do_x is not None: X = np.full(n, float(do_x))          # THE INTERVENTION: cut Z→X
    Y = 2.0*X - 0.5*Z + 60 + 2*rng.standard_normal(n)         # health
    return Z, X, Y

Z, X, Y = world(20000)
naive_slope = ols(X[:, None], Y)[1]
# the interventional ORACLE: force X and measure the response directly
y_do = {x0: world(20000, do_x=x0)[2].mean() for x0 in (2.0, 5.0)}
true_effect = (y_do[5.0] - y_do[2.0]) / 3.0
print(f"naive regression slope of Y on X: {naive_slope:.2f}")
print(f"TRUE causal effect (measured by actually intervening): {true_effect:.2f}")
print("→ the naive slope is more than double the truth: age drives both low exercise and poor")
print("  health, and regression happily launders that path into the X coefficient")

naive regression slope of Y on X: 5.40
TRUE causal effect (measured by actually intervening): 1.98
→ the naive slope is more than double the truth: age drives both low exercise and poor
  health, and regression happily launders that path into the X coefficient


---
### 🕐 Session 2 of 3 — *d-Separation & the Backdoor Adjustment* (~40 min)
**Goal:** block the right paths: adjust for confounders, DON'T adjust for colliders — both verified.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (modern estimators).

---

## 3. Which Variables to Control For

💡 **Intuition.** The graph answers it. Non-causal association flows along **backdoor paths** (X ← Z → Y); conditioning on Z *blocks* them — so regressing Y on X **and Z** recovers the causal slope. But the rule cuts both ways: a **collider** (X → C ← Y) is blocked *by default*, and conditioning on it **opens** a spurious path — 'controlling for everything' is how careful-sounding analyses create bias from thin air (selection on admission, hospitalization, employment...). d-separation is the complete bookkeeping of which paths are open.

In [3]:
# Backdoor adjustment on World 1: add the confounder to the regression
adj_slope = ols(np.c_[X, Z], Y)[1]
print(f"backdoor-adjusted slope (control for age): {adj_slope:.3f}   (truth 2.0 — recovered)")

# World 2: X and Y CAUSALLY UNRELATED, but both cause C (a collider)
def world2(n):
    X2 = rng.standard_normal(n)
    Y2 = rng.standard_normal(n)
    C2 = X2 + Y2 + 0.5*rng.standard_normal(n)              # e.g., 'got admitted' = talent + luck
    return X2, Y2, C2
X2, Y2, C2 = world2(20000)
print(f"\nWorld 2 — X, Y independent by construction:")
print(f"  slope of Y on X (correct: ~0):             {ols(X2[:, None], Y2)[1]:+.3f}")
print(f"  slope of Y on X, 'controlling' for C:      {ols(np.c_[X2, C2], Y2)[1]:+.3f}   ← collider bias, manufactured")
sel = C2 > 1.0
print(f"  slope among selected units (C > 1):        {ols(X2[sel][:, None], Y2[sel])[1]:+.3f}   ← same bias via selection")

backdoor-adjusted slope (control for age): 2.014   (truth 2.0 — recovered)

World 2 — X, Y independent by construction:
  slope of Y on X (correct: ~0):             -0.002
  slope of Y on X, 'controlling' for C:      -0.800   ← collider bias, manufactured
  slope among selected units (C > 1):        -0.490   ← same bias via selection


---
### 🕐 Session 3 of 3 — *Modern Estimators* (~40 min)
**Goal:** IPW, standardization, doubly-robust — three routes to the same do, audited against the oracle.
**Builds on:** Session 2.

---

## 4. Three Roads to the Interventional Answer

💡 **Intuition.** With binary treatment, three standard estimators of $E[Y|do(T{=}1)] - E[Y|do(T{=}0)]$:
1. **Standardization**: model $E[Y|T,Z]$, average over the *whole* Z population.
2. **Inverse propensity weighting (IPW)**: reweight each unit by $1/P(T{=}t|Z)$ — manufacture the randomized trial that nature refused to run.
3. **Doubly robust**: combine both; consistent if *either* model is right — insurance against your own misspecification.
The oracle discipline continues: we simulate the true counterfactuals and grade all three.

In [4]:
# binary-treatment world: treatment probability depends on Z; effect heterogeneous
def world3(n, force=None):
    Z3 = rng.standard_normal(n)
    p_t = 1/(1 + np.exp(-1.5*Z3))                            # sicker (high Z) → more treated
    T = (rng.random(n) < p_t).astype(float) if force is None else np.full(n, float(force))
    Y3 = 1.5*T + 2.0*Z3 - 0.5*T*Z3 + rng.standard_normal(n)  # true ATE = E[1.5 − 0.5 Z] = 1.5
    return Z3, T, Y3

Z3, T, Y3 = world3(40000)
ate_oracle = world3(200000, force=1)[2].mean() - world3(200000, force=0)[2].mean()

naive = Y3[T==1].mean() - Y3[T==0].mean()

# 1) standardization (outcome regression with interaction)
b = ols(np.c_[T, Z3, T*Z3], Y3)
std_est = (b[0] + b[1]*1 + b[2]*Z3.mean() + b[3]*Z3.mean()) - (b[0] + b[2]*Z3.mean())

# 2) IPW with a logistic propensity (fit by Newton in 6 lines)
w_l = np.zeros(2)
Xl = np.c_[np.ones(len(Z3)), Z3]
for _ in range(30):                                          # Newton–Raphson for logistic regression
    p = 1/(1+np.exp(-Xl @ w_l))
    H = (Xl * (p*(1-p))[:, None]).T @ Xl
    w_l += np.linalg.solve(H, Xl.T @ (T - p))
p_hat = 1/(1+np.exp(-Xl @ w_l))
ipw_est = np.mean(T*Y3/p_hat) - np.mean((1-T)*Y3/(1-p_hat))

# 3) doubly robust (AIPW)
mu1 = b[0] + b[1] + (b[2]+b[3])*Z3
mu0 = b[0] + b[2]*Z3
dr_est = np.mean(mu1 - mu0 + T*(Y3-mu1)/p_hat - (1-T)*(Y3-mu0)/(1-p_hat))

print(f"interventional ORACLE ATE: {ate_oracle:+.3f}")
print(f"naive difference in means: {naive:+.3f}   (confounded — way off)")
print(f"standardization:           {std_est:+.3f}")
print(f"IPW:                       {ipw_est:+.3f}")
print(f"doubly robust:             {dr_est:+.3f}")

interventional ORACLE ATE: +1.502
naive difference in means: +3.341   (confounded — way off)
standardization:           +1.490
IPW:                       +1.498
doubly robust:             +1.498


**The honest boundary:** every method above assumed *no unmeasured confounding* — an assumption the data can never certify (that's what makes causal inference hard, and what instrumental variables, sensitivity analysis, and RCTs exist for). The graph tells you what to adjust; only design tells you whether you measured enough.

## 5. Conclusion

Seeing ≠ doing (naive slope 2× the truth, measured); backdoors are blocked by adjustment while colliders are *opened* by it (both manufactured on demand); and standardization/IPW/DR all recover the interventional oracle within noise. Regression answers questions about the world as it is; these tools answer questions about worlds we might make.

---
## Where next

- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — the statistical machinery under each estimator.
- [Uncertainty in ML](./Uncertainty_in_ML.ipynb) — predictions under distribution shift: causality's sibling problem.